# Turn-Taking Fusion Experiments

### Note that Early Fusion will only work for unimodal data -- or data collected at the same Hz!
#### Also please note you will have to configure a lot of the paths yourself
Thin driver for dyadic-fusion experiments on the turn-taking corpus. All heavy lifting (dataset classes, model classes, training helpers, τ-sweep) lives in `fusion_lib.py`. This notebook just:

1. Declares a modality registry (currently set up for CPC and OpenFace).
2. Declares a dict of experiments — each experiment is `{streams, fusion}` where `streams = [{modality, role}, ...]` and `fusion ∈ {'unimodal', 'early', 'neural_concat'}`.
3. Runs each experiment, collects results, and generates a τ-sweep curve.



## Section 0 — Setup

## Init Local

In [ ]:
import os, sys, json, time
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

import fusion_lib as fl
import fusion_runner as fr

# Suppress the pydevd "frame eval" warning that pollutes tqdm output.
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"


In [ ]:
# Dry-run / small-subset mode. Set to None to use all available samples.
# For smoke tests, try MAX_SAMPLES_PER_SPLIT = 200 to verify plumbing end-
# to-end in ~30 s before committing to a full run. Truncation is preceded
# by a seed-deterministic shuffle per split so the subset is representative.
MAX_SAMPLES_PER_SPLIT = None

# Model / training hyperparameters. Defaults match the CSCI-535 practicum
# (multimodal_fusion_practicum.ipynb): GRU_UNITS=64, EPOCHS=30, BATCH_SIZE=32,
# LR=1e-3, patience=4, dropout=0.3, num_layers=3 (EarlyFusion only).

HIDDEN_SIZE       = 64
DROPOUT           = 0.3
NUM_LAYERS_EARLY  = 3            # GRU stack depth for EarlyFusion only
EPOCHS            = 30           # max epochs; early stopping may trigger earlier
BATCH_SIZE        = 32
LEARNING_RATE     = 1e-3
PATIENCE          = 4            # early-stop after N epochs w/o val-F1 gain
NUM_WORKERS       = 2 
SEED              = 42
TRAIN_TAU_MS      = 400

In [ ]:
RUN_ENV = "local"  # "colab" or "local"

if RUN_ENV == "colab":
    from google.colab import drive
    drive.mount('/content/drive/')
    # cd '535_project_data' ## Need to figure this out if Gera wants to run on Colab
    BASE = Path("/content/drive/MyDrive/535_project_data")
    CPC_DIR = BASE / "data/spliced/cpc"
    OPENFACE_DIR = BASE / "data/spliced/openface_new"
    LABELS_DIR = BASE / "data/spliced/labels_tau"
    MANIFEST_PATH = BASE / "data/spliced/manifests/manifest_cpc_available.csv"
    COORDINATION_CSV_PATH = BASE / "data/spliced/coordination/openface/openface/all_coordination_features_tau_0400.csv"
    COORDINATION_CONTINUOUS_DIR = BASE / "data/spliced/coordination/openface_continuous_arrays"
    RESULTS_DIR = BASE / "fusion_runs"

elif RUN_ENV == "local":
    # Kait's local paths
    # BASE = Path("/Users/kaitlinzareno/Desktop/csci535/csci535-project")
    # CPC_DIR = BASE / "data/spliced/cpc"
    # OPENFACE_DIR = BASE / "data/spliced/openface_new"
    # LABELS_DIR = BASE / "data/labels_tau"
    # MANIFEST_PATH = BASE / "data/manifests/manifest_cpc_available.csv"
    # COORDINATION_CSV_PATH = BASE / "data/coordination/openface/all_coordination_features_tau_0400.csv"
    # COORDINATION_CONTINUOUS_DIR = BASE / "data/coordination/openface_continuous_arrays"
    # RESULTS_DIR = BASE / "fusion_runs"

    # Rasika's local paths
    BASE = Path("/Users/rasikaramanan/Documents/usc/by_semester/sp26/csci535/project/seamless/csci535-project")
    CPC_DIR = BASE / "turn_taking_analysis/subset/cpc"
    OPENFACE_DIR = BASE / "turn_taking_analysis/subset/openface_windows"
    LABELS_DIR = BASE / "turn_taking_analysis/model_input/labels"
    MANIFEST_PATH = BASE / "turn_taking_analysis/manifests/manifest.csv"
    COORDINATION_CSV_PATH = BASE / "turn_taking_analysis/subset/coordination/openface/all_coordination_features.csv"
    COORDINATION_CONTINUOUS_DIR = BASE / "turn_taking_analysis/subset/coordination/openface_continuous_arrays"    
    RESULTS_DIR = BASE / "turn_taking_analysis/fusion_runs"
    
TRAIN_LABELS_PATH = LABELS_DIR / "labels_tau_0400.json"

assert not (_missing := [p for p in (BASE, CPC_DIR, OPENFACE_DIR, LABELS_DIR, MANIFEST_PATH, COORDINATION_CSV_PATH, COORDINATION_CONTINUOUS_DIR, TRAIN_LABELS_PATH) if not p.exists()]), f"Missing paths: {_missing}"

OPENFACE_DIM = 23
COORDINATION_CONTINUOUS_DIM = OPENFACE_DIM  # wcc: shape (D = 23, 2 * max_lag = 10 + 1)

# Single notebook-level RUN_ID; all 5 ablations land under one run dir.
RUN_ID = time.strftime("%Y%m%d_%H%M%S")
MANIFEST_TYPE = MANIFEST_PATH.stem  # e.g. 'manifest' or 'poc_manifest'
RUN_ROOT = RESULTS_DIR / MANIFEST_TYPE / RUN_ID
RUN_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
print(f'manifest:         {MANIFEST_PATH}')
print(f'train labels:     {TRAIN_LABELS_PATH}')
print(f'dry-run cap:      {MAX_SAMPLES_PER_SPLIT}')
print(f'run root:        {RUN_ROOT}')


## Section 1 — Modality registry

Each entry: `{dir, feature_dim, frame_rate_hz}`. `frame_rate_hz` is metadata (not consumed by the loader) — kept for documentation. Add new modalities by appending entries and referencing them in experiment configs.

The on-disk layout each modality must satisfy:

```
<modality.dir>/
  <interaction_id>_<participant_id>/
    NNNN.NN-NNNN.NN_<interaction_id>_<participant_id>.npy
    ...
```

### Standard (no coordination)

### + coordination

In [ ]:
MODALITY_REGISTRY = fl.make_modality_registry_coordination({
    "cpc": {
        "kind": "spliced",
        "dir": str(CPC_DIR),
        "feature_dim": 256,
        "frame_rate_hz": 100.0,
    },

    "openface": {
        "kind": "spliced",
        "dir": str(OPENFACE_DIR),
        "feature_dim": 23,
        "frame_rate_hz": 30.0,
    },

    # Alias used by your coordination GRU configs.
    "coordination_openface": {
        "kind": "coordination",
        "coordination_mode": "summary",
        "feature_dim": len(fl.COORDINATION_FEATURE_COLUMNS),
        "frame_rate_hz": 0.0,
    },

    # Explicit summary name.
    "coordination_openface_summary": {
        "kind": "coordination",
        "coordination_mode": "summary",
        "feature_dim": len(fl.COORDINATION_FEATURE_COLUMNS),
        "frame_rate_hz": 0.0,
    },

    # Continuous WCC stream for CSA.
    "coordination_openface_continuous": {
        "kind": "coordination",
        "coordination_mode": "continuous",
        "dir": str(COORDINATION_CONTINUOUS_DIR),
        "feature_dim": COORDINATION_CONTINUOUS_DIM,
        "frame_rate_hz": 0.0,
    },
})

for name, cfg in MODALITY_REGISTRY.items():
    print(f'  {name}: dim={cfg["feature_dim"]}  rate={cfg["frame_rate_hz"]} Hz  dir={cfg["dir"]}')

## Section 2 — Load manifest + label sanity

In [ ]:
manifest_rows = fl.load_manifest(str(MANIFEST_PATH))
print(f'manifest rows: {len(manifest_rows)}')
print(f'splits:        {dict((sp, sum(1 for r in manifest_rows if r["split"] == sp)) for sp in ("train", "val", "test"))}')

# Cross-check label coverage against the manifest.
train_labels = fl.load_labels(str(TRAIN_LABELS_PATH))
samples = fl.enumerate_samples(manifest_rows, train_labels)
summary = fl.summarize_samples(samples)
print(f'\ntotal training-usable samples at τ=400 ms: {summary["total"]}')
print(f'per-split: {summary["per_split"]}')
for split, dist in summary['per_split_class'].items():
    print(f'  {split:5s}  class distribution: {dist}')

## Section 3 — Experiment definitions

Each experiment declares its streams (modality + role) and fusion family.

**Fusion families supported:**
- `unimodal` — single stream, single GRU + FC.
- `early` — N streams concatenated on the feature axis, single 3-layer GRU.
- `neural_concat` — exactly 2 streams, two parallel GRUs, concat final hidden states, FC head.

**Roles supported per stream:**
- `speaker` — the floor holder (perspective participant).
- `listener` — the co-participant (resolved via manifest).

Add / remove experiments by editing `EXPERIMENTS` below.

DEFINITIONS:
- standard:     fl.run_experiment              + no attention args
- ssa:          fl.run_experiment              + attention args
- coordination: fl.run_experiment_coordination + no attention args
- csa:          fl.run_experiment_coordination + attention args

### Standard (no coordination, GRU)

In [ ]:
EXPERIMENTS = {
    'cpc_both_early': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'early',
    },
    'cpc_both_neural_concat': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'neural_concat',
    },
    'cpc_speaker_of_listener': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'neural_concat',
    },
    'cpc_speaker_of_dyad': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
    'full_dyad': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'cpc',      'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'neural_concat',
    }
}

print(f'configured experiments: {len(EXPERIMENTS)}')
for name, cfg in EXPERIMENTS.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

### Standard + Self Attention (SSA)

In [ ]:
ATTENTION_EXPERIMENTS = {
    'cpc_both_attention': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'of_speaker_of_listener_attention': {
        'streams': [
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'cpc_speaker_of_listener_attention': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion':  'self_attention',
    },
    'cpc_speaker_of_dyad_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
        ],
        'fusion': 'self_attention',
    },
    'full_dyad_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'cpc',      'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'self_attention',
    }
    }


### Standard + Cross Attention (SCA) / Self+Cross Attention (SSCA)

Cross-attention requires exactly 2 streams. Two model variants:
- `cross_attention`: bidirectional cross-modality attention only (no self-attn)
- `self_cross_attention`: per-stream self-attention, then cross-attention (combined)

Both directly comparable to `cpc_speaker_of_listener_attention` in the SSA block above.

In [ ]:
CROSS_ATTENTION_EXPERIMENTS = {
    'cpc_speaker_of_listener_cross_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'cross_attention',
    },
    'cpc_speaker_of_listener_self_cross_attention': {
        'streams': [
            {'modality': 'cpc',      'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
        ],
        'fusion': 'self_cross_attention',
    },
}

print(f'configured cross-attention experiments: {len(CROSS_ATTENTION_EXPERIMENTS)}')
for name, cfg in CROSS_ATTENTION_EXPERIMENTS.items():
    print(f'  {name:55s}  fusion={cfg["fusion"]:25s}  streams={cfg["streams"]}')

### Experiments w/ Coordination

In [ ]:
EXPERIMENTS_COORDINATION ={
     'cpc_speaker_plus_coordination': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'coordination_openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
    
    #ONLY TEST OPENFSACE AND COORD LOCAL
        "openface_with_coord_neural_concat": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        "fusion": "neural_concat",
    },

    #ONLY TEST OPENFSACE AND COORD LOCAL
      "openface_BOTH_with_coord_neural_concat": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        "fusion": "neural_concat",
    },

    'cpc_speaker_of_listener_coord': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        'fusion':  'neural_concat',
    },
     
     'cpc_speaker_both_faces_coord': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {"modality": "coordination_openface", "role": "speaker"},
        ],
        'fusion':  'neural_concat',
    },

    'full_dyad_plus_coordination': {
        'streams': [
            {'modality': 'cpc', 'role': 'speaker'},
            {'modality': 'cpc', 'role': 'listener'},
            {'modality': 'openface', 'role': 'speaker'},
            {'modality': 'openface', 'role': 'listener'},
            {'modality': 'coordination_openface', 'role': 'speaker'},
        ],
        'fusion': 'neural_concat',
    },
}

print(f'configured experiments: {len(EXPERIMENTS_COORDINATION)}')
for name, cfg in EXPERIMENTS_COORDINATION.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

### Coodination + Self Attention (CSA)

Note: 
- attention_heads_by_modality tells the SelfAttention class how many self-attention heads to make for that specific modality.
- Since coordination is small, will only use 2 heads. Uses 4 heads for other modalities  

MUST be in order:
- ex. for cpc_speaker_coord_self_attention, 
- streams = 
    - "modality": "cpc", "role": "speaker"
    - "modality": "coordination_openface_continuous", "role": "speaker"
- attention_heads_by_modality = [4,2] s,t. 4 --> CPC, speaker and 2 --> coordination, speaker

In [ ]:
ATTENTION_EXPERIMENTS_COORDINATION = {
    "cpc_speaker_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 2], 
    },

    "cpc_speaker_cpc_listener_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "cpc", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "cpc_speaker_of_speaker_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "openface", "role": "speaker"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "openface_coord_self_attention": {
        "streams": [
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "cpc_speaker_of_listener_coord_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 2],
    },

    "full_dyad_plus_coordination_self_attention": {
        "streams": [
            {"modality": "cpc", "role": "speaker"},
            {"modality": "cpc", "role": "listener"},
            {"modality": "openface", "role": "speaker"},
            {"modality": "openface", "role": "listener"},
            {"modality": "coordination_openface_continuous", "role": "speaker"},
        ],
        "fusion": "self_attention",
        "attention_heads_by_modality": [4, 4, 4, 4, 2],
    },
}

print(f'configured experiments: {len(ATTENTION_EXPERIMENTS_COORDINATION)}')
for name, cfg in ATTENTION_EXPERIMENTS_COORDINATION.items():
    print(f'  {name:30s}  fusion={cfg["fusion"]:14s}  streams={cfg["streams"]}')

## Section 3.5 — Preflight

Validates every configured experiment against the current data + manifest BEFORE running any of them. Catches `torch.stack` shape divergences (including the continuous-coord-zero-fallback hazard that crashed the τ-sweep on `cpc_speaker_coord_self_attention`), missing feature files, model-build issues (`attention_dim % heads`, mixed-rate early fusion, etc.), and stream/fusion config errors — all up-front, with full provenance (interaction_id, speaker/listener participants, split, window timestamps, candidate paths, observed/expected shapes) per failing sample.

Runs at full sample-set scope regardless of what `MAX_SAMPLES_PER_SPLIT` is set to. The audit's truncated subset and the runner's truncated subset are not nested under the same seed (audit broadens the class filter to cover τ-sweep), so smoke-scope audit can produce false PASSes. Full-scope header reads finish in ~1 minute on local SSD.

Inspect the diagnostics; rerun this cell after data fixes; proceed to Section 4 only when satisfied. The cell sets `PREFLIGHT_OK` for convenience but does NOT auto-gate Section 4.

In [ ]:
# Load the coordination summary CSV once across all five ablation calls so
# the per-ablation preflight doesn't re-parse it. Only the coordination /
# csa ablations actually use it; passing it through the others is harmless
# (those experiments don't have summary-coord streams).
coord_lookup = None
if COORDINATION_CSV_PATH is not None and Path(str(COORDINATION_CSV_PATH)).exists():
    coord_lookup = fl.load_coordination_features(
        str(COORDINATION_CSV_PATH),
        feature_cols=fl.COORDINATION_FEATURE_COLUMNS,
    )

# Shared header cache so paths referenced by multiple ablations
# (e.g. CPC speaker streams used by all four) are header-read exactly once.
shared_header_cache = {}

ablation_specs = [
    ("standard",     EXPERIMENTS,                       False),  # GRU, no coord
    ("ssa",          ATTENTION_EXPERIMENTS,             False),  # SSA, no coord
    ("sca",          CROSS_ATTENTION_EXPERIMENTS,       False),  # cross-attn, no coord
    ("coordination", EXPERIMENTS_COORDINATION,          True),   # GRU + coord
    ("csa",          ATTENTION_EXPERIMENTS_COORDINATION, True),  # SSA + coord
]

preflight_reports = {}

_pf_t0 = time.time()
for ablation_name, experiments_dict, uses_coord in ablation_specs:
    preflight_reports[ablation_name] = fl.preflight_experiments(
        ablation_kind=ablation_name,
        experiments=experiments_dict,
        modality_registry=MODALITY_REGISTRY,
        manifest_rows=manifest_rows,
        labels_path=str(TRAIN_LABELS_PATH),
        coordination_csv_path=str(COORDINATION_CSV_PATH) if uses_coord else None,
        coordination_lookup=coord_lookup if uses_coord else None,
        missing_coordination="zeros",
        max_samples_per_split=None,        # full-corpus audit; see markdown above
        seed=SEED,
        hidden_size=HIDDEN_SIZE,
        dropout=DROPOUT,
        num_layers_early=NUM_LAYERS_EARLY,
        attention_dim=None,                # falls back to hidden_size in build_model
        attention_heads=4,                 # global default; per-exp overrides honored
        attention_layers=2,
        attention_pooling="mean",
        verbose=True,
        header_cache=shared_header_cache,
    )

print(f"\nPreflight finished in {time.time() - _pf_t0:.1f}s\n")

# --- Aggregate summary ----------------------------------------------------
total_pass = 0
total_fail = 0
all_failed = []

for ablation_name, report in preflight_reports.items():
    s = report["summary"]
    print(
        f"[{ablation_name:13s}] "
        f"experiments={s['total_experiments']:>2}  "
        f"PASS={s['passed']:>2}  FAIL={s['failed']:>2}  "
        f"cfg_err={s['total_config_errors']:>3}  "
        f"shape_err={s['total_shape_errors']:>5}  "
        f"missing={s['total_missing_files']:>5}  "
        f"synth_fwd_fail={s['total_synthetic_forward_failures']:>2}"
    )
    total_pass += s["passed"]
    total_fail += s["failed"]
    for exp_name, exp_report in report["experiments"].items():
        if exp_report["status"] == "FAIL":
            all_failed.append((ablation_name, exp_name, exp_report))

print(f"\nOverall: PASS={total_pass}, FAIL={total_fail}")

# --- Per-FAIL detail (skip PASS — they're fine) ---------------------------
for ablation_name, exp_name, exp_report in all_failed:
    print(f"\n=== FAIL: {ablation_name}/{exp_name} ===")

    if exp_report["config_errors"]:
        print("  config_errors:")
        for e in exp_report["config_errors"]:
            print(f"    - {e}")

    if exp_report["synthetic_forward_error"]:
        print(f"  synthetic_forward_error: {exp_report['synthetic_forward_error']}")

    bad_streams = [
        ss for ss in exp_report.get("stream_summaries", [])
        if ss["stack_hazard"] or ss["shape_mismatch_count"] > 0 or ss["missing_count"] > 0
    ]
    if bad_streams:
        print("  per-stream summary (only streams with issues shown):")
        for ss in bad_streams:
            print(
                f"    [{ss['stream_idx']}] {ss['modality']}/{ss['role']} "
                f"(kind={ss['kind']}, expected_T={ss['expected_T']}, "
                f"expected_F={ss['expected_F']})"
            )
            print(f"        observed_shapes: {ss['observed_shapes']}")
            print(
                f"        missing={ss['missing_count']}, "
                f"shape_mismatch={ss['shape_mismatch_count']}, "
                f"stack_hazard={ss['stack_hazard']}"
            )

    by_kind = defaultdict(list)
    for e in exp_report.get("missing_file_errors", []) + exp_report.get("shape_errors", []):
        by_kind[e["error_kind"]].append(e)

    if by_kind:
        print("  errors (up to 5 per kind, with provenance):")
        for kind, records in by_kind.items():
            print(f"    {kind} ({len(records)} total):")
            for e in records[:5]:
                print(
                    f"      iid={e['interaction_id']}, "
                    f"speaker={e['speaker_participant']}, "
                    f"listener={e['listener_participant']}, "
                    f"split={e['split']}, "
                    f"window=[{e['window_start_s']:.2f},{e['window_end_s']:.2f}]s, "
                    f"stream={e['stream_modality']}/{e['stream_role']}"
                )
                if e.get("observed_shape") is not None or e.get("expected_shape") is not None:
                    print(
                        f"        observed={e.get('observed_shape')}, "
                        f"expected={e.get('expected_shape')}"
                    )
                if e.get("tried_paths"):
                    print(f"        tried: {e['tried_paths'][0]}")
                    if len(e["tried_paths"]) > 1:
                        print(f"               (+{len(e['tried_paths']) - 1} more candidate path(s))")
            if len(records) > 5:
                print(f"      ... {len(records) - 5} more {kind} record(s) suppressed")

# --- Gate flag (informational; does NOT auto-gate Section 4) --------------
PREFLIGHT_OK = (total_fail == 0)
print(f"\nPREFLIGHT_OK = {PREFLIGHT_OK}")
if PREFLIGHT_OK:
    print("Safe to proceed to Section 4.")
else:
    print(
        "Inspect FAIL detail above before running Section 4. Section 4 is "
        "NOT auto-gated on this flag — re-run this cell after fixing data "
        "or set PREFLIGHT_OK = True manually to override after diagnosis."
    )

## Section 4 — Run experiments

Trains one model per experiment at the canonical τ (default 400 ms), keeps the best-val-macro-F1 checkpoint, and evaluates on test. Early stopping kicks in after `PATIENCE` epochs without a val-F1 improvement (caps training well under `EPOCHS`). Results are accumulated in `results` and reused in Section 5 for the τ-sweep (input features are τ-invariant; only labels change).

#### General Runner

All driver logic lives in [fusion_runner.py](fusion_runner.py). Each ablation block below trains every experiment, rebuilds the best checkpoint, runs the per-τ detailed evaluation, and persists artifacts incrementally — so a crash after experiment N still leaves N completed `<exp>/results.json` + `model_state.pt` on disk.


#### run standard

In [ ]:
standard_detailed, standard_out_dir = fr.run_ablation_block(
    ablation_name="standard",
    experiments=EXPERIMENTS,
    use_coordination=False,
    use_attention=False,
    out_root=RUN_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    labels_dir=LABELS_DIR,
    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
    train_tau_ms=TRAIN_TAU_MS,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)


#### run ssa

In [ ]:
ssa_detailed, ssa_out_dir = fr.run_ablation_block(
    ablation_name="ssa",
    experiments=ATTENTION_EXPERIMENTS,
    use_coordination=False,
    use_attention=True,
    out_root=RUN_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    labels_dir=LABELS_DIR,
    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
    train_tau_ms=TRAIN_TAU_MS,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)


#### run cross attention (SCA / SSCA)

In [ ]:
sca_detailed, sca_out_dir = fr.run_ablation_block(
    ablation_name="sca",
    experiments=CROSS_ATTENTION_EXPERIMENTS,
    use_coordination=False,
    use_attention=True,
    out_root=RUN_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    labels_dir=LABELS_DIR,
    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
    train_tau_ms=TRAIN_TAU_MS,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)


#### run coordination

In [ ]:
coordination_detailed, coordination_out_dir = fr.run_ablation_block(
    ablation_name="coordination",
    experiments=EXPERIMENTS_COORDINATION,
    use_coordination=True,
    use_attention=False,
    out_root=RUN_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    labels_dir=LABELS_DIR,
    tau_grid_ms=(400,),
    train_tau_ms=TRAIN_TAU_MS,
    coordination_csv_path=COORDINATION_CSV_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)


#### run csa

In [ ]:
csa_detailed, csa_out_dir = fr.run_ablation_block(
    ablation_name="csa",
    experiments=ATTENTION_EXPERIMENTS_COORDINATION,
    use_coordination=True,
    use_attention=True,
    out_root=RUN_ROOT,
    modality_registry=MODALITY_REGISTRY,
    manifest_rows=manifest_rows,
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    labels_dir=LABELS_DIR,
    tau_grid_ms=fl.DEFAULT_TAU_GRID_MS,
    train_tau_ms=TRAIN_TAU_MS,
    coordination_csv_path=COORDINATION_CSV_PATH,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
    num_layers_early=NUM_LAYERS_EARLY,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    seed=SEED,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
)


## Section 5 — Cross-ablation rollup

Writes a single `index.json` at `RUN_ROOT` summarizing all 5 ablations. Per-ablation artifacts already landed inside each ablation block above — this is the run-level summary.


In [ ]:
fr.write_run_index(
    run_root=RUN_ROOT,
    ablation_results={
        "standard":     standard_detailed,
        "ssa":          ssa_detailed,
        "sca":          sca_detailed,
        "coordination": coordination_detailed,
        "csa":          csa_detailed,
    },
    manifest_path=MANIFEST_PATH,
    train_labels_path=TRAIN_LABELS_PATH,
    train_tau_ms=TRAIN_TAU_MS,
    tau_grid_ms_by_ablation={
        "standard":     fl.DEFAULT_TAU_GRID_MS,
        "ssa":          fl.DEFAULT_TAU_GRID_MS,
        "sca":          fl.DEFAULT_TAU_GRID_MS,
        "coordination": (400,),
        "csa":          fl.DEFAULT_TAU_GRID_MS,
    },
    modality_registry=MODALITY_REGISTRY,
    global_hyperparams={
        "hidden_size": HIDDEN_SIZE,
        "dropout": DROPOUT,
        "num_layers_early": NUM_LAYERS_EARLY,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "patience": PATIENCE,
        "num_workers": NUM_WORKERS,
        "seed": SEED,
        "max_samples_per_split": MAX_SAMPLES_PER_SPLIT,
    },
    device=fl.resolve_device("auto"),
)

print(f"Run directory: {RUN_ROOT}")
for p in sorted(RUN_ROOT.iterdir()):
    print(f"  {p.name}")
